# Phase 3 step 3 — Retraining ablations (2 runs)

Same Mask R-CNN R50-FPN recipe as the Phase 2 baseline (6,000 iterations), changing ONE thing per run:

| Run | Name | Change | Time on T4 |
|---|---|---|---|
| A | `r50_clahe_disk` | images: solar-disk masking + CLAHE contrast (train AND predict) | ~3 min preprocessing + ~1.3 h |
| B | `r50_allann` | labels: every annotator's version is a training sample (921 instead of 566) | ~1.3 h |

Each run ends by saving **raw predictions** (val + test) to `MyDrive/solar_filament/Phase 3/`, exactly like
step 1. Scoring and submission CSVs are then done on the local computer — no validation cells here.

**Order:** run setup (sections 1-4), then Run A, then Run B. They are independent: you can run only one, or run
them in two separate sessions.

**If Colab disconnects:** run sections 1-4 again, then re-run the cell of the run that was interrupted. Training
resumes from the last checkpoint on Drive (every 1,000 iterations); preprocessing is redone automatically
(~3 min, it is kept on Colab's local disk, not Drive).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
OUT_DIR = f"{DRIVE_DIR}/output"            # checkpoints (persistent)
PHASE3_DIR = f"{DRIVE_DIR}/Phase 3"         # raw predictions to send back
os.makedirs(PHASE3_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
os.environ["SOLAR_PREPROC_DIR"] = "/content/preprocessed"   # fast local disk, regenerable
print("Drive ready:", OUT_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5-8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.is_available())

## 4. Quick checks (~30 s) and a helper to save raw predictions to Drive

In [ ]:
!python -m pytest -q -p no:warnings tests/test_preprocess.py tests/test_official_metric.py
import shutil
def save_raw(name):
    for split in ("val", "test"):
        src = f"{OUT_DIR}/raw_preds_{name}_{split}.json.gz"
        shutil.copy(src, PHASE3_DIR)
        print(split, os.path.getsize(src) // 1024, "KB ->", PHASE3_DIR)

## Run A — solar-disk masking + CLAHE (`r50_clahe_disk`)
Preprocessing writes enhanced copies of all 887 images (~3 min), then training (~1.3 h), then raw predictions
on the enhanced val/test images. A warning that some RPN head weights were skipped is expected (as in Phase 2).

In [ ]:
NAME_A = "r50_clahe_disk"
!python run_pipeline.py --step preprocess --preproc clahe_disk
!python run_pipeline.py --step train --name {NAME_A} --preproc clahe_disk
!python run_pipeline.py --step dump  --name {NAME_A} --preproc clahe_disk
save_raw(NAME_A)

## Run B — all annotators' labels (`r50_allann`)
Original images; every annotator set of a training file becomes its own training sample.

In [ ]:
NAME_B = "r50_allann"
!python run_pipeline.py --step train --name {NAME_B} --label-policy all
!python run_pipeline.py --step dump  --name {NAME_B}
save_raw(NAME_B)

## 5. Send results back
Download the raw prediction files of the run(s) you finished (also in `MyDrive/solar_filament/Phase 3/`) into
the local folder `Result Phase 3/`, then tell Claude. The training loss curves are in
`MyDrive/solar_filament/output/detectron2/<name>/metrics.json` if needed.

In [ ]:
from google.colab import files
for name in ("r50_clahe_disk", "r50_allann"):
    for split in ("val", "test"):
        f = f"{PHASE3_DIR}/raw_preds_{name}_{split}.json.gz"
        if os.path.exists(f):
            files.download(f)
        else:
            print("not found (run not finished?):", f)